 #### Contact jvamaraju@nvidia.com or sugandhas@nvidia.com or aasthaj@nvidia.com for questions.

# Data Curation for DAPT (Domain Adaptive Pre-Training)

## Learning Goals and Datasets

This playbook aims to demonstrate how to curate data from various data sources to customize foundation models through domain-adaptive pre-training and improve performance on domain-specific tasks.

In this playbook, we will leverage chip domain/hardware datasets from open-source GitHub repositories (`./github_repos_opensource.jsonl`), wiki URLs (`./wikiurls_opensource.jsonl`), and academic papers(`/datasets/pdfs/`). 

This playbook utilizes specific tools and techniques. First, we convert all files to Txt format (if not already in Txt), compress files on disk, add metadata, and convert them to JSON. Then, we leverage [NeMo Curator](https://github.com/NVIDIA/NeMo-Curator/tree/main) to mine high-quality text at scale from a massive code-generation corpus. [NeMo Curator](https://github.com/NVIDIA/NeMo-Curator/tree/main), built on Dask and RAPIDS, is instrumental in scaling data curation and providing GPU acceleration. We use its capabilities to extract text, identify code file types, fix unicode errors, filter quality through heuristics, deduplicate, and redact personal information. We finally also provide steps to blend and shuffle data sources for continued pre-training.


## NeMo Tools and Resources

* [NeMo-Curator GitHub repo](https://github.com/NVIDIA/NeMo-Curator/tree/main)

## Software Requirements
* This playbook has been tested on: nvcr.io/nvidia/nemo:24.03.01.framework. It is expected to work similarly on other environments. Within the container, from the NeMo-Curator directory install using `pip install --extra-index-url https://pypi.nvidia.com ".[cuda12x]"`
* NeMo Curator currently requires Python 3.10 and the GPU accelerated modules require CUDA 12 or above installed in order to be used.

## Hardware Requirements
* This playbook can run on CPUs or GPUs. For GPUs, this playbook has been tested on minimum 1xA100 80G

### Overview of steps involved: 
In this notebook, we will use the datasets in the `dapt-data-curation/datasets` folder to illustrate data curation through this pipeline. Specifically sample data collected in:
* `/datasets/github_repos`
* `/datasets/pdfs_txt` (Pdfs converted to txt, see `./data_curation_pdf.ipynb`)
* `/datasets/wiki` (we extract data from htmls, parse convert to txt and store compressed files)

The notebook follows the steps below:<br>
- Step 1: Install requirements and import libraries<br>
- Step 2: Find all text files in the given directory tree<br>
- Step 3: Examine the identified files (optional) <br>
- Step 4: Copy all files we want to include and compress them<br>
- Step 5: Convert GitHub files and pdfs to JSON format and store metadata (Using NeMo curator)<br>
- Step 6: Download text from wiki urls, extract metadata and convert to JSON (Using NeMo curator)<br>
- Step 7: Process and filter data with Nemo Curator<br>
    - File type identification and separation
    - Document-level exact deduplication
    - Heuristic-based quality filtering (Number of lines, worc count, top N-grams, etc.)
    - Fix unicode errors via ftfy
    - PII redaction
- Step 8: Blend datasets and shuffle

## Step 1: Install requirements and import libraries

In [7]:
# install all required packages for data curation if required
! pip3 install -r ../requirements.txt 

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 653.6/653.6 kB 38.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.7/145.7 kB 85.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 105.7 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 293.0 MB/s eta 0:00:00


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 291.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 kB 285.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 268.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 272.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 310.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.3/155.3 kB 317.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 441.8/441.8 kB 143.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 335.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.3/300.3 kB 161.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 307.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.8/341.8 kB 312.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.9/140.9 kB 336.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.9/152.9 kB 336.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 345.1 MB/s eta 0:00:00
  Created wheel for cchardet: filename=cchardet-2.1.7-cp310-cp310-linux_x86_64.whl size=289369 sha256=06f65daca9058cb55d764d35b9a9ff6ffd55591da352771ac5586b8910ecf6c6
  Stored in directory: /tmp/pip-ephem-wheel-cache-f8qpj05j/wheels/ee/e0/ab/e01326f15c59438d080b1496dbab8091e952ec72f35e3c437e
  Created wheel for docx2txt: filename=docx2txt-0.8-py3-none-any.whl size=3960 sha256=59dd2ab8f5ff28e096399c1d2f654f2b1aeda822765078de2483e5c0a32150ae
  Stored in directory: /tmp/pip-ephem-wheel-cache-f8qpj05j/wheels/22/58/cf/093d0a6c3ecfdfc5f6ddd5524043b88e59a9a199cb02352966
  Created wheel for python-p

In [1]:
import argparse
import os
import shutil
from typing import Any
import glob
import gzip
import pandas as pd
from nemo_dc_dapt.helpers import write_jsonl, count_lines
# from filters import IncompleteStoryFilter
from nemo_dc_dapt.modifiers import QuotationUnifier
from nemo_dc_dapt.utils import clean_and_unify, filter_dataset, filter_code, filter_code_dataset, redact, dedupe
from nemo_dc_dapt.utils import FilterFilesBasedOnLines_txt, FilterFilesBasedOnLines_code
from nemo_dc_dapt.docbuilder import DAPTtxtDownloader

import nemo_curator as nc
from nemo_curator import ScoreFilter, Sequential
from nemo_curator.datasets import DocumentDataset
from nemo_curator.filters import RepeatingTopNGramsFilter, WordCountFilter
from nemo_curator.modifiers.pii_modifier import PiiModifier
from nemo_curator.modifiers.unicode_reformatter import UnicodeReformatter
from nemo_curator.modules import ExactDuplicates
from nemo_curator.modules.modify import Modify
from nemo_curator.utils.distributed_utils import get_client
from nemo_curator.utils.file_utils import get_all_files_paths_under, separate_by_metadata
from nemo_curator.utils.script_utils import add_distributed_args

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2: Find all text files in the given directory tree

Now we will run the `create_manifest.py` script that takes in a directory as input, finds all the text files in the directory tree, and produces a manifest csv file containing a list of file names (file paths), file sizes, and number of lines in each of the text files found in the directory tree. 
- Input1: `directory`: the path to the input collection directory that has the data we want to search through
- Input2: `manifest_file`: the path to the output csv file
- output: `manifest.csv`: a csv file with three columns: Path (listing file names), Size (listing file sizes) and Lines (listing number of lines in each file) is saved to the  `dapt-data-curation/output` folder

#### An exmaple of a sample output showing the manifest csv file is shown below. 

![manifest](imgs/manifest_csv.png)

 A list of open source github repos relevant to chip design can be found in `github_repos_opensource.jsonl` file.
 A list of wiki urls relevant to chip design can be found in `wikiurls_opensource.jsonl` file.
 In this playbook, we will sample datasets from a few of these github repos, wiki urls and relevant academic papers.
 

In [2]:
# specify path to the input directory that has the data that we want to search through
directory = "../../../datasets"
SCRIPT_DIR_PATH = os.path.dirname(os.path.abspath('./')) #current path
DATA_DIR = '../../../datasets/'
DATA_DIR_wiki = '../../../datasets/wiki' # path to wiki files
DATA_DIR_github = '../../../datasets/github_repos' # path to github repos

#convert pdfs to txt files using pdfcleaner (see data_curation_pdf.ipynb)
DATA_DIR_pdftext = '../../../datasets/pdfs_txt' #path to pdf converted to txt files

JSONL_ROOT_DIR = os.path.join(DATA_DIR, "jsonl") #output path for converted json files
PARQUET_ROOT_DIR = os.path.join(DATA_DIR, "parquet") #output path for converted parquet files

# For the purposes of this tutorial, we will use the smaller datasets to demonstrate the curation pipeline.

In [3]:
# create a folder for storing output files under the dapt-data-curation directory
! mkdir ../../../datasets/output
! mkdir ../../../datasets/output_pdf

mkdir: cannot create directory ‘../../../datasets/output’: File exists
mkdir: cannot create directory ‘../../../datasets/output_pdf’: File exists


In [4]:
# specify the path to (and the name of) the output csv file
manifest_file  = "../../../datasets/output/manifest.csv"
# specify the path to (and the name of) the output csv file for pdf txts
manifest_file_pdf  = "../../../datasets/output_pdf/manifest.csv"

In [5]:
# make sure the above folders do not already exist. New files will not be created otherwise. 
# create the manifest csv file with a list of all text files found. This csv will have three columns listing file names, their sizes and number of lines in each of them
! bash ../create_manifest.sh $DATA_DIR_github $manifest_file

# create the manifest csv file with a list of all pdf text files found. This csv will have three columns listing file names, their sizes and number of lines in each of them
! bash ../create_manifest_non_tree.sh $DATA_DIR_pdftext $manifest_file_pdf

Output file already exists: ../../../datasets/output/manifest.csv
Output file already exists: ../../../datasets/output_pdf/manifest.csv


#### This is what the `dapt-data-curation/output` folder should look like at the end of Step 2:

![pipeline](imgs/output_step2.png)

## Step3 (optional): Examine the identified files

Here, we will run `report_manifest.py` that enables us to examine the largest files from the indetified files listed in the manifest csv. It takes the manifest csv as input and reports the top 10 biggest files along with their corresponding percentile sizes. It allows visualization through text based histograms for various file types. 

- Input: `manifest.csv`: a csv file with three columns: Path (listing file names), Size (listing file sizes) and Lines (listing number of lines in each file)
- Output:<br> 
       - Top 10 biggest files along with their corresponding percentile sizes <br>
       - List of the top 10 biggest files with their paths, number of lines and file size<br>
       - List of files types greater than 5MB in size and number of such files for each file type <br>
       - Histograms for various file types showing the distribution of file sizes
- Output: `file_size_histogram.png`: saves the file size histogram in the `dapt-data-curation/output` folder

#### An example of a file size histogram output is shown below:

![histogram](imgs/file_size_histogram.png)



In [6]:
# run the report_manifest.py script, with the manifest csv file as an argument passed to the script
#! python3 ../report_manifest.py $manifest_file 

# run the report_manifest.py script, with the manifest csv file as an argument passed to the script
! python3 ../report_manifest.py $manifest_file_pdf

10 Files
Top 10 percentiles (Size_MB):
0.90    0.059761
0.91    0.061157
0.92    0.062553
0.93    0.063950
0.94    0.065346
0.95    0.066743
0.96    0.068139
0.97    0.069536
0.98    0.070932
0.99    0.072329
Name: Size_MB, dtype: float64
Top 10 files (size):
/workspace/datasets/pdfs_txt/ViraEye_Energy-Efficient_Stereo_Vision_Accelerator_with_Binary_Neural_Network_in_55_nm_CMOS.pdf.txt.gz : 184 Lines 0.004962MB
/workspace/datasets/pdfs_txt/decc.pdf.txt.gz : 451 Lines 0.021462MB
/workspace/datasets/pdfs_txt/AutoFlex_Unified_Evaluation_and_Design_Framework_for_Flexible_Hybrid_Electronics.pdf.txt.gz : 578 Lines 0.022931MB
/workspace/datasets/pdfs_txt/K-Means_Hashing_An_Affinity-Preserving_Quantization_Method_for_Learning_Binary_Compact_Codes.pdf.txt.gz : 686 Lines 0.0243MB
/workspace/datasets/pdfs_txt/image_retrieval.pdf.txt.gz : 666 Lines 0.028986MB
/workspace/datasets/pdfs_txt/A_bundle_approach_to_efficient_MAP-inference_by_Lagrangian_relaxation.pdf.txt.gz : 959 Lines 0.030178MB
/worksp

### This is what the `dapt-data-curation/output` folder should look like at the end of Step 3:

![output_3](imgs/output_step3.png)

## Step 4: Copy all files we want to include and compress them

Here we will run `execute_manifest.py` which takes a manifest csv as input and copies over all the files listed in the csv, gzips them and calculates the md5sum. We can either use the original manifest csv if we want to include all the text files we found in the directory tree for DAPT training. 

Alternatively, we can also use any of the filtered manifest csv files generated in the previous step in we want to selectivley use a particular category of files for DAPT training. 

In this example, we just use the original manifest csv to illustrate processing of a larger dataset. 

- Input: `manifest.csv`: a csv file with three columns: Path (listing file names), Size (listing file sizes) and Lines (listing number of lines in each file)
- Output: manifest folder containing each file listed in the input csv gzipped (.gz). In addition this folder contains a .txt file listed the origianl paths from where each of these files was copied. The manifest folder is saved in the `dapt-data-curation/output` folder.

In [7]:
# run the execute_manifest.py script, with the manifest csv file as an argument passed to the script
! python3 ../execute_manifest.py $manifest_file
! python3 ../execute_manifest.py $manifest_file_pdf

Done.
Done.


#### This is what the `dapt-data-curation/output` folder should look like at the end of Step 4:

![output_5](imgs/output_step5.png)

#### This is what the `dapt-data-curation/output/manifest` folder should look like at the end of Step 4:

![output_5_manifest](imgs/manifest_folder.png)

#### This is what the `dapt-data-curation/output/manifest/info.txt` file should look like at the end of Step 4:

![output_5_manifest_info](imgs/manifest_folder_txt.png)

## Step 5: Convert GitHub files and pdfs to JSON format and store metadata (Using NeMo curator)

In [8]:
def get_local_dir_path(orig_name):
    # get basename of orig_name (no extensions)
    basename = os.path.splitext(os.path.basename(orig_name))[0]
    # get directory name of orig_name
    dirname = os.path.dirname(orig_name)
    # local directory path is dirname/basename_collected
    local_dir = os.path.join(dirname, basename)
    return local_dir

#Read files from 'info.txt' in each manifest folders, extract metadata and convert to JSON
info_git = pd.read_csv(get_local_dir_path(manifest_file)+'/info.txt', skiprows=[0],header = None, names = ['file_name','orig_path'])
info_git['file_name'] = get_local_dir_path(manifest_file) + '/' + info_git['file_name']
info_pdf = pd.read_csv(get_local_dir_path(manifest_file_pdf)+'/info.txt', skiprows=[0],header = None, names = ['file_name','orig_path'])
info_pdf['file_name'] = get_local_dir_path(manifest_file_pdf) + '/' + info_pdf['file_name']
info = pd.concat([info_git, info_pdf], ignore_index=True)

for index, row in info.iterrows():
#     print(row["file_name"], row["orig_path"])
    write_jsonl(row["file_name"], JSONL_ROOT_DIR, row["orig_path"])

Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/896656f025d3b5139df5578950acbce8.spice.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/af5ef796675eabfbba415fe900745818.cdl.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/fdc76fade0fe5e10b9061b3bfe3116c6.cdl.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/84d91da4500e8be6fc02b922d05438dd.lef.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Conve

Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/7ffefbe6afb9c940f01c79dde78ea24c.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/f74f3551b2c8f3c7b4f7f21a1c3dcb92.spice.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/a1932dfb1cde71af5300fbc53c02751d.tsv.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/78c8dafc35e27a6fd84594977636fbdf.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Convertin

Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/b4694702cab925d4cfd5133762bc8495.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/13c0dbe98c65638197262a7f29545699.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/b86d88a7f81953bcfb2bb6b7caf2b653.lef.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/8ba0672d565e84c46f8b6da9966c07c5.lef.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting 

Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/b28e5b26f19fe5c19794a305eadf7e5f.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/33a6574c91b74ce8ec97aaf29679f6c3.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/83d3642d33b05c58030fa4431dc0e77f.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/1ee424eb6651e4873be7d65e7ab928e6.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../

Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/44ed58e63ea0ae1ded4c01eba5aeb466.spice.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/170e823570b946de9367e37017bedeaa.lef.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/ba25423cf6667dfa87f61bf58b0b5087.lef.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/7dddcc0a609031cc37396af611abd521.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Convert

...

Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/1a61dd6149d6b3f6062a4ac7b93daaec.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/87b0605ed925e8e98eabfac3259f5813.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/72d38aa4acfd6613e3c37e841e78dbc0.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/e9188de2101f16cf18605be6cb98caa0.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../

Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/84c313d91b8bbb0b49325096fbbfb67d.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/d1bb641137d73c8f6dbbb29aa9ea0d2a.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/4ef0cc3ee64d2d6f7af0997b687350bf.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/651567124ef70003d6e0764d12c09a64.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../

Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/563c53e4c306b71fd3689d05d3ec2281.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/0d4489e61f6f6663e6f2e5430652a214.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/6fb04bb73e0a63d6c386ab8f92bb60ef.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/5e0f69d7ad5e73022fb8c3fed7d0a664.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../

Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/11e00a9243adee76c776e726e2b4d860.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/ccfa143d4fd89f36cf7de07d3f1724f2.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/77d4b000aeecf13e5768366c181a099d.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/cae5e45f664e0a0d83b2c5514726827b.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../

Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/5a580031364a0f510e7e1c2a1fe99ef3.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/9b2699eaa67023ae82abda034e078a53.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/167cb4c44c40c90ce101e8226189fe48.v.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/output/manifest/9ff1501c42e24e8f5d1690725b5858c5.emf.gz' to JSONL format (output: '../../../datasets/jsonl')...
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '.

## Step 6: Download text from wiki urls, extract metadata and convert to JSON (Using NeMo curator)

In [9]:
"""
Downloads text from Wiki and converts it to JSONL format.

Creates the directory path where the JSONL files are saved.
"""
# Download the Wiki data
downloader = DAPTtxtDownloader(DATA_DIR_wiki)

urls = pd.read_json(path_or_buf='./wikiurls_opensource.jsonl', lines=True)
urls = urls[0].tolist()

#Process first 100 wiki urls
for url in urls[:100]:
    wiki_val_fp = downloader.download(url)
    # Convert to JSONL files.
    write_jsonl(wiki_val_fp, JSONL_ROOT_DIR, "/"+os.path.abspath(wiki_val_fp))

Download directory:  ../../../datasets/wiki
File '../../../datasets/wiki/Memory%20rank.wiki.gz' already exists, skipping download.
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/wiki/Memory%20rank.wiki.gz' to JSONL format (output: '../../../datasets/jsonl')...
File '../../../datasets/wiki/Isochronous%20signal.wiki.gz' already exists, skipping download.
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/wiki/Isochronous%20signal.wiki.gz' to JSONL format (output: '../../../datasets/jsonl')...
File '../../../datasets/wiki/A73.wiki.gz' already exists, skipping download.
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/wiki/A73.wiki.gz' to JSONL format (output: '../../../datasets/jsonl')...
File '../../../datasets/wiki/CCIX.wiki.gz' already exists, skipping download.
Output directory '../../../da

Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/wiki/Loop%20interchange.wiki.gz' to JSONL format (output: '../../../datasets/jsonl')...
File '../../../datasets/wiki/Iterative%20Viterbi%20decoding.wiki.gz' already exists, skipping download.
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/wiki/Iterative%20Viterbi%20decoding.wiki.gz' to JSONL format (output: '../../../datasets/jsonl')...
File '../../../datasets/wiki/Expm.wiki.gz' already exists, skipping download.
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/wiki/Expm.wiki.gz' to JSONL format (output: '../../../datasets/jsonl')...
File '../../../datasets/wiki/Wilson%20current%20mirror.wiki.gz' already exists, skipping download.
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datase

Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/wiki/Classic%20RISC%20pipeline.wiki.gz' to JSONL format (output: '../../../datasets/jsonl')...
File '../../../datasets/wiki/Coherency%20Granule.wiki.gz' already exists, skipping download.
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/wiki/Coherency%20Granule.wiki.gz' to JSONL format (output: '../../../datasets/jsonl')...
File '../../../datasets/wiki/Marvell.wiki.gz' already exists, skipping download.
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/wiki/Marvell.wiki.gz' to JSONL format (output: '../../../datasets/jsonl')...
File '../../../datasets/wiki/PS-ON%20Signal.wiki.gz' already exists, skipping download.
Output directory '../../../datasets/jsonl' is not empty. Adding conversion to JSONL.
Converting '../../../datasets/wiki/PS-ON%20Sign

## Step 7: Process, clean and filter data with NeMo Curator

In [10]:
# Arguments required for dask client 
#see nemo_curator.utils.script_utils.add_distributed_args

class Args:
    n_workers = os.cpu_count()
    threads_per_worker = 1
    protocol = 'tcp'
    files_per_partition = 2
    device = 'gpu' #or 'cpu'
    scheduler_address = None
    scheduler_file = None
    
args = Args()
args.n_workers = min(args.n_workers, 8)

In [11]:
# Initialize the Dask cluster.
client = get_client(args, args.device)
print(f"Running curation pipeline on '{JSONL_ROOT_DIR}'...")

# Read data from json files(JSONL_ROOT_DIR) and add it to DocumentDataset (main dataset class) for efficient processing
files = [
    fp
    for fp in get_all_files_paths_under(JSONL_ROOT_DIR, recurse_subdirectories=False)
    if fp.endswith(".jsonl")
]
print("Reading the data...")
orig_dataset = DocumentDataset.read_json(files, add_filename=True)
dataset = orig_dataset

#separate data into text and code files 
separated_data = separate_by_metadata(dataset.df,JSONL_ROOT_DIR, 'file_type').compute()

text_files = [
    fp
    for fp in get_all_files_paths_under(JSONL_ROOT_DIR+"/text", recurse_subdirectories=False)
    if fp.endswith(".jsonl")
]

code_files = [
    fp
    for fp in get_all_files_paths_under(JSONL_ROOT_DIR+"/code", recurse_subdirectories=False)
    if fp.endswith(".jsonl")
]

# Read text and code files separately to apply different filters
orig_dataset_text = DocumentDataset.read_json(text_files, add_filename=True)
orig_dataset_code = DocumentDataset.read_json(code_files, add_filename=True)

# Define data curation steps for text files
curation_steps_text = Sequential(
    [
        dedupe,
        FilterFilesBasedOnLines_txt,
        filter_dataset,
        clean_and_unify,
#         redact_pii, #Not needed for non-code files
    ]
)

# Define data curation steps for code files
curation_steps_code = Sequential(
    [
        dedupe,
        FilterFilesBasedOnLines_code,
        filter_code,
        clean_and_unify,
        redact
    ]
)

dataset_text = curation_steps_text(orig_dataset_text)
dataset_code = curation_steps_code(orig_dataset_code)

print("Executing the pipeline...")
dataset_text = dataset_text.persist()
dataset_code = dataset_code.persist()

print(f"Original dataset length for text files: {len(orig_dataset_text.df)}")
print(f"After dataprep: {len(dataset_text.df)}")

print(f"Original dataset length for code files: {len(orig_dataset_code.df)}")
print(f"After dataprep: {len(dataset_code.df)}")

print("Writing the results to disk...")

#save filtered and cleaned datasets
# Overwrite existing files and save all jsonl in the curated directory.
out_path = os.path.join(JSONL_ROOT_DIR, "curated")
if os.path.isdir(out_path):
    shutil.rmtree(out_path)
os.makedirs(out_path)
dataset_text.to_json(out_path, write_to_filename=True)
dataset_code.to_json(out_path, write_to_filename=True)

# Split the dataset by file category and save curated files
separated_data_text = separate_by_metadata(dataset_text.df,out_path, 'category').compute()
separated_data_code = separate_by_metadata(dataset_code.df,out_path, 'category').compute()

# # Overwrite existing files and save to all parquet format in the curated directory if needed 
out_path = os.path.join(PARQUET_ROOT_DIR, "curated")
if os.path.isdir(out_path):
    shutil.rmtree(out_path)
os.makedirs(out_path)
dataset_text.to_parquet(out_path, write_to_filename=True)
dataset_code.to_parquet(out_path, write_to_filename=True)

client.close()

Torch is using RMM memory pool
Running curation pipeline on '../../../datasets/jsonl'...
Reading the data...
Reading 9489 files
Reading 292 files
Reading 9197 files


2024-05-29 01:11:42,756 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 4be04591c1c4a23bd1316c1ce4eac46e initialized by task ('shuffle-transfer-4be04591c1c4a23bd1316c1ce4eac46e', 99) executed on worker tcp://127.0.0.1:44929
2024-05-29 01:11:44,290 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 4be04591c1c4a23bd1316c1ce4eac46e deactivated due to stimulus 'task-finished-1716945104.289393'
2024-05-29 01:12:03,979 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 1b6d9d9bff864056d12fa879fc14e181 initialized by task ('shuffle-transfer-1b6d9d9bff864056d12fa879fc14e181', 152) executed on worker tcp://127.0.0.1:45285
2024-05-29 01:12:54,528 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 1b6d9d9bff864056d12fa879fc14e181 deactivated due to stimulus 'task-finished-1716945174.5255938'
2024-05-29 01:13:55 INFO:Loaded recognizer: EmailRecognizer
2024-05-29 01:13:55 INFO:Loaded recognizer: PhoneRecognizer
2024-05-29 01:13:55 INFO:Loaded recognizer: Spac

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.7/587.7 MB 117.5 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.5.0
    Uninstalling typing_extensions-4.5.0:
      Successfully uninstalled typing_extensions-4.5.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albumentations 1.4.4 requires numpy>=1.24.4, but you have numpy 1.24.3 which is incompatible.
cudf 23.12.0 requires pandas<1.6.0dev0,>=1.3, but you have pandas 2.0.1 which is incompatible.
cudf 23.12.0 requires protobuf<5,>=4.21, but you have protobuf 3.20.3 which is incompatible.
cugraph 23.12.0 requires dask-cuda==23.12.*, but you have dask-cuda 24.4.0 which is incompatible.
cugraph 23.12.0 requires rapids-dask-dependency==23.12.*, but you have rapids-dask-dependency 24.4.1 which is incompatible.
cugraph-service-server 23.12.0 requires dask-cuda==23.12.*, but you have dask-cuda 24.4.0 which is incompatible.
cugraph-service-server 23.12.0 requires rapids-dask-dependency==23.12.*, but you have rapids-dask-dependency 24.4.1 which is incompatible.
cuml 23.12.0 requires dask-cuda==23.12.*, but you have dask-cuda 24.4

2024-05-29 01:14:11 INFO:Finished downloading model en_core_web_lg
2024-05-29 01:14:18 INFO:Loaded recognizer: EmailRecognizer
2024-05-29 01:14:18 INFO:Loaded recognizer: PhoneRecognizer
2024-05-29 01:14:18 INFO:Loaded recognizer: SpacyRecognizer
2024-05-29 01:14:18 INFO:Loaded recognizer: UsSsnRecognizer
2024-05-29 01:14:18 INFO:Loaded recognizer: CreditCardRecognizer
2024-05-29 01:14:18 INFO:Loaded recognizer: IpRecognizer
2024-05-29 01:14:18 WARNING:model_to_presidio_entity_mapping is missing from configuration, using default
2024-05-29 01:14:18 WARNING:low_score_entity_names is missing from configuration, using default
2024-05-29 01:14:26,934 - distributed.scheduler - ERROR - Couldn't gather keys: {('getitem-1dd8b49d07feed881caa1c46ccb2ebf2', 0): 'forgotten'}
2024-05-29 01:14:26,935 - distributed.client - WARNING - Couldn't gather 1 keys, rescheduling (('getitem-1dd8b49d07feed881caa1c46ccb2ebf2', 0),)
2024-05-29 01:14:27,074 - distributed.scheduler - ERROR - Couldn't gather keys: {

...

Original dataset length for text files: 292
After dataprep: 267
Original dataset length for code files: 9197
After dataprep: 8834
Writing the results to disk...


/usr/local/lib/python3.10/dist-packages/nemo_curator/utils/distributed_utils.py:379: UserWarning: Empty partition found
  warnings.warn(f"Empty partition found")
/usr/local/lib/python3.10/dist-packages/nemo_curator/utils/distributed_utils.py:379: UserWarning: Empty partition found
  warnings.warn(f"Empty partition found")
/usr/local/lib/python3.10/dist-packages/nemo_curator/utils/distributed_utils.py:379: UserWarning: Empty partition found
  warnings.warn(f"Empty partition found")
/usr/local/lib/python3.10/dist-packages/nemo_curator/utils/distributed_utils.py:379: UserWarning: Empty partition found
  warnings.warn(f"Empty partition found")


Writing to disk complete for 292 partitions
Writing to disk complete for 8834 partitions


/usr/local/lib/python3.10/dist-packages/nemo_curator/utils/distributed_utils.py:379: UserWarning: Empty partition found
  warnings.warn(f"Empty partition found")
/usr/local/lib/python3.10/dist-packages/nemo_curator/utils/distributed_utils.py:379: UserWarning: Empty partition found
  warnings.warn(f"Empty partition found")
/usr/local/lib/python3.10/dist-packages/nemo_curator/utils/distributed_utils.py:379: UserWarning: Empty partition found
  warnings.warn(f"Empty partition found")
/usr/local/lib/python3.10/dist-packages/nemo_curator/utils/distributed_utils.py:379: UserWarning: Empty partition found
  warnings.warn(f"Empty partition found")


Writing to disk complete for 292 partitions
Writing to disk complete for 8834 partitions


## Step 8: Blend datasets and shuffle

In [12]:
root_path = os.path.join(JSONL_ROOT_DIR, "curated")
dataset_paths = [root_path+"/CPP", root_path+"/VerilogVHDL", root_path+"/text", root_path+"/Python"]
dataset_weights = [1.0, 4.0, 4.0, 1.0]
target_size = 1000
output_path = JSONL_ROOT_DIR+"/data_blended"

# Blend the datasets
datasets = [DocumentDataset.read_json(path) for path in dataset_paths]
blended_dataset = nc.blend_datasets(target_size, datasets, dataset_weights)

shuffle = nc.Shuffle(seed=42)
blended_dataset = shuffle(blended_dataset)

# Save the blend
blended_dataset.to_json(output_path)

Reading 377 files
Reading 3588 files
Reading 267 files
Reading 16 files
Writing to disk complete for 1000 partitions


###  We are now ready to use these curated data files for continued pre-training of LLMs